In [1]:
import os
import glob
import tiktoken
import numpy as np
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sklearn.manifold import TSNE
import plotly.graph_objects as go
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_huggingface import HuggingFaceEmbeddings
import gradio as gr


C:\Users\mohsi\AppData\Local\Temp\ipykernel_13016\3365189895.py:8: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, TextLoader


In [11]:
MODEL = "openai/gpt-oss-120b"
DB_NAME = "vector_db"
base_url = "https://api.groq.com/openai/v1"

load_dotenv(override=True)
groq_api_key = os.getenv('GROQ_API_KEY')
if groq_api_key:
    print(f"GROQ API Key exists and begins {groq_api_key[:8]}")
else:
    print("GROQ API Key not set")

GROQ API Key exists and begins gsk_RVLO


In [12]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [13]:
vectorstore = Chroma(persist_directory=DB_NAME, embedding_function=embeddings)

### Set up the 2 key LangChain objects: retriever and llm

#### A sidebar on "temperature":
- Controls how diverse the output is
- A temperature of 0 means that the output should be predictable
- Higher temperature for more variety in answers

Some people describe temperature as being like 'creativity' but that's not quite right
- It actually controls which tokens get selected during inference
- temperature=0 means: always select the token with highest probability
- temperature=1 usually means: a token with 10% probability should be picked 10% of the time

Note: a temperature of 0 doesn't mean outputs will always be reproducible. You also need to set a random seed. We will do that in weeks 6-8. (Even then, it's not always reproducible.)

Note 2: if you want creativity, use the System Prompt!

In [14]:
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 3})
llm = ChatOpenAI(model_name=MODEL, temperature=0, base_url = base_url, api_key = groq_api_key)

In [15]:
retriever.invoke("who is Abdul Mateen?")

[Document(id='924a3f78-4a01-4de4-816a-ca916e6ae984', metadata={'source': 'knowledge-base\\alumni\\academia.md', 'doc_type': 'alumni'}, page_content='---\n\n## Academic Alumni in Pakistani Higher Education\n\n### 1. Returning Faculty at PUCIT / FCIT\nMany current faculty members at FCIT are themselves alumni of PUCIT who completed doctoral research under prestigious scholarships (Fulbright, Erasmus Mundus, HEC Overseas) before returning to teach:\n* **Dr. Saadia Shahzad**: Assistant Professor, Department of Computer Science (completed PhD research at PUCIT under co-supervision with senior professors, specializing in automatic pattern recognition).\n* **Mr. Abdul Mateen**: Assistant Professor, Department of Computer Science (long-serving faculty alumnus and educator).\n* **Dr. Madiha Khalid**: Assistant Professor, Department of Software Engineering (collaborating on cloud computing and distributed data systems).\n* Numerous lecturers and lab instructors across the four departments gradua

In [16]:
llm.invoke("Who is Abdul Mateen?")

AIMessage(content='**Abdul\u202fMateen** is a name that can refer to several different people, so the exact identity depends on the context in which you encountered it. Below are a few of the more commonly referenced individuals with that name:\n\n| Field | Notable Person Named Abdul\u202fMateen | Brief Description |\n|-------|-----------------------------------|-------------------|\n| **Cricket** | **Abdul\u202fMateen (born 1999)** | A Pakistani cricketer who has played at the domestic level for teams such as Karachi Whites and Sindh. He is a right‑hand batsman and occasional off‑spin bowler. |\n| **Politics / Public Service** | **Abdul\u202fMateen (Bangladeshi politician)** | A member of the Bangladesh Awami League who has served in local government positions (e.g., upazila chairman) in the Dhaka Division. |\n| **Academia / Research** | **Dr. Abdul\u202fMateen** | A scholar in the field of Islamic studies or South Asian history who has published articles on contemporary religious mov

In [20]:
SYSTEM_PROMPT = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""

In [21]:
def answer_question(question: str, history):
    docs = retriever.invoke(question)
    context = "\n".join([doc.page_content for doc in docs])
    system_prompt = SYSTEM_PROMPT.format(context=context)
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
    return response.content

In [23]:
gr.ChatInterface(answer_question, title="University Chatbot", description="Ask questions about Insurellm.").launch()

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.
